In [1]:
import pandas as pd

In [2]:
# df = pd.read_excel('Dataset/financial_loan_data.xlsx')
# df.to_csv('financial_data.csv',index=False)

In [3]:
df = pd.read_csv('Dataset/financial_data.csv')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38576 entries, 0 to 38575
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     38576 non-null  int64  
 1   address_state          38576 non-null  object 
 2   application_type       38576 non-null  object 
 3   emp_length             38576 non-null  object 
 4   emp_title              37138 non-null  object 
 5   grade                  38576 non-null  object 
 6   home_ownership         38576 non-null  object 
 7   issue_date             38576 non-null  object 
 8   last_credit_pull_date  38576 non-null  object 
 9   last_payment_date      38576 non-null  object 
 10  loan_status            38576 non-null  object 
 11  next_payment_date      38576 non-null  object 
 12  member_id              38576 non-null  int64  
 13  purpose                38576 non-null  object 
 14  sub_grade              38576 non-null  object 
 15  te

In [7]:
df['issue_date'] = pd.to_datetime(df['issue_date'], format='mixed')

In [8]:
df['Year'] = df['issue_date'].dt.year
df['Month'] = df['issue_date'].dt.month
df['Month_Name'] = df['issue_date'].dt.month_name()
df['Day_Name'] = df['issue_date'].dt.day_name()

In [5]:
# Current Month
MTD = 12
# Previous Month
PMTD = 11

In [25]:
## Total Loan Application
total_loan_application = df['id'].count()
MTD_Loan_Application = df[df['Month'] == MTD]['id'].count()
PMTD_Loan_Application = df[df['Month'] == PMTD]['id'].count()
MoM_Loan_Amount_Growth = ((MTD_Loan_Application-PMTD_Loan_Application)/PMTD_Loan_Application)*100

In [26]:
print(f'Total Loan Applications: {total_loan_application}')
print(f'Month-to-Date Applications: {MTD_Loan_Application}')
print(f'Previous Month-to_Date Applications: {PMTD_Loan_Application}')
print(f'Month-over-Month Growth: {MoM_Loan_Amount_Growth:,.2f}%')

Total Loan Applications: 38576
Month-to-Date Applications: 4314
Previous Month-to_Date Applications: 4035
Month-over-Month Growth: 6.91%


In [29]:
## total funded amount
total_funded_amount = df['loan_amount'].sum()
MTD_total_funded_amount = df[df['Month'] == MTD]['loan_amount'].sum()
PMTD_total_funded_amount = df[df['Month'] == PMTD]['loan_amount'].sum()
MoM_funded_amount_groth = ((MTD_total_funded_amount - PMTD_total_funded_amount)/PMTD_total_funded_amount)*100

In [31]:
print(f'Total Funded Amount: Rs.{total_funded_amount}')
print(f'Month-to-Date Funded Amount: Rs.{MTD_total_funded_amount}')
print(f'Previous Month-to-Date Funded Amount: Rs.{PMTD_total_funded_amount}')
print(f'Month-over-Month Growth: {MoM_funded_amount_groth:,.2f}%')

Total Funded Amount: Rs.435757075
Month-to-Date Funded Amount: Rs.53981425
Previous Month-to-Date Funded Amount: Rs.47754825
Month-over-Month Growth: 13.04%


In [32]:
df['loan_status'].unique()

array(['Charged Off', 'Fully Paid', 'Current'], dtype=object)

In [36]:
good_loan = df['loan_status'].isin(['Fully Paid', 'Current'])
bad_loan = df['loan_status'].isin(['Charged Off'])

In [39]:
Good_Loan_Details = df[good_loan]
Bad_Loan_Details = df[bad_loan]

In [44]:
good_loan_applicants = Good_Loan_Details['id'].count()
bad_loan_applicants = Bad_Loan_Details['id'].count()

print(f'Good Loan Applicants: {good_loan_applicants}')
print(f'Bad Loan Applicants: {bad_loan_applicants}')

Good Loan Applicants: 33243
Bad Loan Applicants: 5333


In [46]:
# good loan and bad loan applicants
good_loan_funded_amount = Good_Loan_Details['loan_amount'].sum()
good_loan_funded_amount

np.int64(370224850)

In [47]:
bad_loan_funded_amount = Bad_Loan_Details['loan_amount'].sum()
bad_loan_funded_amount

np.int64(65532225)

In [62]:
good_loan_details = Good_Loan_Details.groupby('loan_status').agg(
    total_loan_applicants = ('id', 'count'),
    total_loan_amount = ('loan_amount', 'sum'),
    total_amount_received = ('total_payment', 'sum')
).reset_index()
good_loan_details

,loan_status,total_loan_applicants,total_loan_amount,total_amount_received
0,Current,1098,18866500,24199914
1,Fully Paid,32145,351358350,411586256


In [63]:
def format_currency(amount):
    return f"Rs.{amount:,.2f}"

In [64]:
# good_loan_details['total_loan_amount'] = good_loan_details['total_loan_amount'].map("Rs.{:,.2f}".format)

In [65]:
columns = ['total_loan_amount', 'total_amount_received']

for col in columns:
    good_loan_details[col] = good_loan_details[col].apply(format_currency)

In [66]:
good_loan_details

,loan_status,total_loan_applicants,total_loan_amount,total_amount_received
0,Current,1098,"Rs.18,866,500.00","Rs.24,199,914.00"
1,Fully Paid,32145,"Rs.351,358,350.00","Rs.411,586,256.00"
